# Istanbul Rail Network: Dynamic Passenger Density Optimization
### Progress Update Report

**Objective:** 
Pivoting from our initial spatial bus routing model, our current focus is on rail-based public transport (Metro, Marmaray, Tram). The primary objective is to optimize dispatch scheduling (headways) to minimize passenger density on platforms and inside trains, ensuring rapid crowd clearance (flushing rate).

**Current Milestone:**
Establishing the baseline hardware constraints and generating dynamic, time-dependent weight coefficients for each station based on turnstile inflow data.

### 1. Hardware Constraints & Fleet Capacities
Before optimizing flow, we mapped out the physical constraints of the network. This includes rolling stock types, set capacities (dependent on platform lengths), and average line speeds.

In [ ]:
import pandas as pd

# Load the hardware specifications for the rail network
df_hardware = pd.read_csv('istanbul_rayli_sistem_verileri_guncel.csv')
df_hardware.head()

### 2. Mathematical Model: Time-Dependent Station Proportions
To optimize schedules dynamically, the model needs to know which stations are the most critical at any given hour. We calculate the relative weight of a station $i$ at time $t$ against the total line traffic at time $t$.

$$W_{i,t} = \frac{P_{i,t}}{\sum_{j=1}^{N} P_{j,t}}$$

Where:
* $P_{i,t}$ = Total passengers entering station $i$ at hour $t$.
* $N$ = Total number of stations on the specific line.

In [ ]:
import time

# Data Processing Script for Station Weights
start_time = time.time()
print("1. Reading the massive CSV file...")

# Memory optimization using categorical types for massive datasets (IBB Turnstile Data)
df_turnstile = pd.read_csv("rayli_sistem_temiz.csv", dtype={'hat_adi': 'category', 'durak_adi': 'category'})

print("2. Grouping data and aggregating passenger flow...")
df_saatlik_profil = df_turnstile.groupby(['hat_adi', 'durak_adi', 'saat'], observed=True)['toplam_yolcu'].sum().reset_index()
df_saatlik_profil = df_saatlik_profil[df_saatlik_profil['toplam_yolcu'] > 0]

print("3. Calculating time-dependent proportions (Weights)...")
hat_saat_toplam = df_saatlik_profil.groupby(['hat_adi', 'saat'], observed=True)['toplam_yolcu'].transform('sum')
df_saatlik_profil['weight'] = df_saatlik_profil['toplam_yolcu'] / hat_saat_toplam

sonuc_df = df_saatlik_profil[['hat_adi', 'durak_adi', 'saat', 'weight']].copy()
sonuc_df['weight'] = sonuc_df['weight'].round(4)

# Save the output for the optimization model
sonuc_df.to_csv('saatlik_durak_agirliklari.csv', index=False)
print(f"Process completed in {round(time.time() - start_time, 2)} seconds.")

### 3. Output Validation
Viewing the calculated weights. For example, verifying the rush-hour load shifts between residential and commercial hubs.

In [ ]:
# Load and display the generated weights
df_weights = pd.read_csv('saatlik_durak_agirliklari.csv')
df_weights.head(10)

### Next Steps
1. **Cost Function Formulation:** Integrating these hourly weights with the hardware constraints (Train Set Capacity) to calculate the minimum headway required to keep density below $D_{max}$.
2. **Simulation:** Simulating the crowd flushing rate for peak bottlenecks (e.g., Levent at 18:00).